# Financial News Sentiment and Stock Returns

Can the text of financial news predict stock and market returns? This notebook compares two NLP approaches on ~20,500 Wall Street Journal articles (2017–2020) tagged with S&P 500 tickers:

1. **Pretrained FinBERT** (`yiyanghkust/finbert-tone`) scoring each article's sentiment.
2. **Frozen BERT embeddings + MLP** trained to predict the mentioned stock's next-day return, tuned with expanding-window time-series cross-validation.

Both signals are evaluated on a locked 2020 holdout at the article level (down / neutral / up) and aggregated monthly against S&P 500 returns.

See the repository README for data setup and results.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Folder containing news.csv and price.csv
DATA_DIR = os.environ.get("DATA_DIR", "data")
OUTPUT_DIR = "outputs"    # intermediate files (FinBERT scores)
FIGURE_DIR = "figures"    # plots used in the README
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)
SENTIMENT_PATH = os.path.join(OUTPUT_DIR, "news_w_sentiment.csv")

def save_fig(name):
    """Save the current matplotlib figure into FIGURE_DIR."""
    plt.savefig(os.path.join(FIGURE_DIR, name), dpi=150, bbox_inches="tight")


In [ ]:
# read dataset
news_df = pd.read_csv(os.path.join(DATA_DIR, 'news.csv'))
price_df = pd.read_csv(os.path.join(DATA_DIR, 'price.csv'))


In [ ]:
news_df.head()

In [ ]:
news_df.shape

In [ ]:
price_df.head()

In [ ]:
price_df['ticker'].unique()

In [ ]:
date_range = news_df['publication_datetime'].min(), news_df['publication_datetime'].max()
print(f"Date range of publication_datetime: {date_range}")

In [ ]:
date_range = price_df['Date'].min(), price_df['Date'].max()
print(f"Date range in price_df: {date_range}")

In [ ]:
# Convert publication_datetime to datetime format
news_df['publication_datetime'] = pd.to_datetime(news_df['publication_datetime'])

# Group by month and count the number of news articles
monthly_news_count = news_df.resample('ME', on='publication_datetime').size()
month_labels = monthly_news_count.index.strftime('%Y-%m')
month_positions = range(len(monthly_news_count))

# Plotting
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(month_positions, monthly_news_count.values, color='skyblue')
ax.set_title('Number of News Articles per Month')
ax.set_xlabel('Publication Month')
ax.set_ylabel('Number of Articles')
ax.set_xticks(list(month_positions))
ax.set_xticklabels(month_labels, rotation=45, ha='right')
ax.grid(axis='y')
plt.tight_layout()
save_fig('articles_per_month.png')
plt.show()


In [ ]:
# Convert publication_datetime to datetime format
news_df['publication_datetime'] = pd.to_datetime(news_df['publication_datetime'])

# Group by week and count the number of news articles
weekly_news_count = news_df.resample('W', on='publication_datetime').size()
week_labels = weekly_news_count.index.strftime('%Y-%m-%d')

# Plotting
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(range(len(weekly_news_count)), weekly_news_count.values, color='skyblue')
ax.set_title('Number of News Articles per Week')
ax.set_xlabel('Publication Week')
ax.set_ylabel('Number of Articles')
ax.set_xticks(range(0, len(weekly_news_count), 10))
ax.set_xticklabels(week_labels[::10], rotation=45, ha='right')
ax.grid(axis='y')
plt.tight_layout()
plt.show()


In [ ]:
# Convert publication_datetime to datetime format
news_df['publication_datetime'] = pd.to_datetime(news_df['publication_datetime'])

# Filter articles that mention AAPL
aapl_articles = news_df[news_df['tickers'].apply(lambda x: 'AAPL' in x)]

# Group by month and count the number of news articles about AAPL
monthly_aapl_news_count = aapl_articles.resample('ME', on='publication_datetime').size()
month_labels = monthly_aapl_news_count.index.strftime('%Y-%m')
month_positions = range(len(monthly_aapl_news_count))

# Plotting
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(month_positions, monthly_aapl_news_count.values, color='skyblue')
ax.set_title('Number of News Articles about AAPL per Month')
ax.set_xlabel('Publication Month')
ax.set_ylabel('Number of Articles')
ax.set_xticks(list(month_positions))
ax.set_xticklabels(month_labels, rotation=45, ha='right')
ax.grid(axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Convert publication_datetime to datetime format
news_df['publication_datetime'] = pd.to_datetime(news_df['publication_datetime'])

# Filter articles that mention AMD
amd_articles = news_df[news_df['tickers'].apply(lambda x: 'AMD' in x)]

# Group by month and count the number of news articles about AMD
monthly_amd_news_count = amd_articles.resample('ME', on='publication_datetime').size()
month_labels = monthly_amd_news_count.index.strftime('%Y-%m')
month_positions = range(len(monthly_amd_news_count))

# Plotting
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(month_positions, monthly_amd_news_count.values, color='skyblue')
ax.set_title('Number of News Articles about AMD per Month')
ax.set_xlabel('Publication Month')
ax.set_ylabel('Number of Articles')
ax.set_xticks(list(month_positions))
ax.set_xticklabels(month_labels, rotation=45, ha='right')
ax.grid(axis='y')
plt.tight_layout()
plt.show()

## Pretrained Large Language Model: FinBERT

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch

# Load FinBERT model and tokenizer: https://huggingface.co/yiyanghkust/finbert-tone
finbert_tokenizer = BertTokenizer.from_pretrained('yiyanghkust/finbert-tone')
finbert = BertForSequenceClassification.from_pretrained('yiyanghkust/finbert-tone', use_safetensors=True)
finbert_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
finbert.to(finbert_device)
finbert.eval()

In [ ]:
# print the maximum length of input tokens
print(f"The maximum length of input tokens for FinBERT: {finbert.config.max_position_embeddings}")

### Handling the 512-token limit: head + tail truncation

BERT-family encoders (including FinBERT) cap input at 512 tokens. Our WSJ
article bodies routinely exceed that, so we can't just pass the raw string.

**Strategy: head + tail truncation (Sun et al. 2019).** Keep the first 128
tokens and the last 382 tokens of the text, dropping the middle. This was the
empirically best strategy on IMDb and Sogou News in the paper
*"How to Fine-Tune BERT for Text Classification?"* (arXiv:1905.05583),
beating head-only, tail-only, and hierarchical chunk-and-pool approaches.

The intuition fits WSJ articles well: the lede carries the headline facts, the
tail often carries analyst commentary / forward guidance / quotes — both ends
tend to matter more than the middle. As an extra gain, we prepend the article
*title* to the body before truncation, since titles are short (~10-20 tokens)
and carry strong sentiment signal.


In [ ]:
# Head+tail tokenization helper (Sun et al. 2019, arXiv:1905.05583).
# Builds a max_length-token batch by keeping head_n tokens from the start and
# tail_n tokens from the end of each text (reserving 2 positions for [CLS] and
# [SEP]). Works with any BERT/RoBERTa-family WordPiece tokenizer.

HEAD_N = 128   # Sun et al. optimum
TAIL_N = 382   # 128 + 382 + 2 (CLS, SEP) = 512

def head_tail_tokenize(texts, tok, head_n=HEAD_N, tail_n=TAIL_N, max_length=512):
    """Tokenize a list of strings with head+tail truncation.

    Returns a dict of {input_ids, attention_mask, token_type_ids} as torch
    tensors, ready to pass to a BERT model as **kwargs.
    """
    assert head_n + tail_n + 2 <= max_length, "head_n + tail_n must leave room for CLS/SEP"
    cls_id = tok.cls_token_id
    sep_id = tok.sep_token_id
    pad_id = tok.pad_token_id if tok.pad_token_id is not None else 0

    all_ids, all_attn = [], []
    for text in texts:
        # Tokenize WITHOUT special tokens and WITHOUT truncation.
        content = tok.encode(text if text else '[NO_TEXT]',
                             add_special_tokens=False, truncation=False)
        if len(content) <= max_length - 2:
            kept = content                              # short enough, no truncation
        else:
            kept = content[:head_n] + content[-tail_n:] # head+tail
        ids = [cls_id] + kept + [sep_id]
        attn = [1] * len(ids)
        pad_len = max_length - len(ids)
        ids.extend([pad_id] * pad_len)
        attn.extend([0] * pad_len)
        all_ids.append(ids)
        all_attn.append(attn)

    return {
        'input_ids':      torch.tensor(all_ids,  dtype=torch.long),
        'attention_mask': torch.tensor(all_attn, dtype=torch.long),
        # Single-sequence classification -> token_type_ids all zeros
        'token_type_ids': torch.zeros(len(texts), max_length, dtype=torch.long),
    }

# Quick sanity check on a long synthetic input
_probe = "this is a test. " * 400   # ~1600 tokens, well over 512
_enc = head_tail_tokenize([_probe], finbert_tokenizer)
print(f"head_tail_tokenize sanity check: input_ids shape = {tuple(_enc['input_ids'].shape)}, "
      f"non-pad tokens = {int(_enc['attention_mask'].sum().item())}")


In [ ]:
news_df['title'] = news_df['title'].fillna('')
news_df['body']  = news_df['body'].fillna('')

# Text variants for easy comparison
news_df['text_title']    = news_df['title']
news_df['text_body']     = news_df['body']
news_df['text_combined'] = (news_df['title'] + ' [SEP] ' + news_df['body']).str.strip()

# Default to title + body, truncated via head+tail per Sun et al. 2019.
TEXT_COL = 'text_combined'  # options: 'text_title', 'text_body', 'text_combined'
print(f"Running FinBERT on: {TEXT_COL} with head+tail truncation ({HEAD_N} head + {TAIL_N} tail)")

def predict_sentiment_batch(texts, batch_size=16):
    """Batched FinBERT inference with Sun et al. 2019 head+tail truncation."""
    rows = []
    cleaned = [t if (isinstance(t, str) and t.strip()) else '[NO_TEXT]' for t in texts]

    for i in range(0, len(cleaned), batch_size):
        batch = cleaned[i:i + batch_size]
        inputs = head_tail_tokenize(batch, finbert_tokenizer,
                                    head_n=HEAD_N, tail_n=TAIL_N, max_length=512)
        inputs = {k: v.to(finbert_device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = finbert(**inputs)
            probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()
        rows.extend(probs)

    return pd.DataFrame(rows, columns=['neutral', 'positive', 'negative'])


### Diagnostic: how much content does head+tail actually keep?

Head+tail is a *fixed* 128+382 truncation — it keeps the same 510 content tokens whether the article is 600 tokens or 6,000. Before trusting it on this corpus we measure (a) the token-length distribution of our articles and (b) the fraction of content actually retained per article. The numbers below drive whether head+tail alone is sufficient or whether a chunk-pool fallback would earn its complexity.

In [ ]:
# DIAGNOSTIC: measure real token-length distribution and head+tail content loss.
# Runs once before FinBERT inference so the decision is evidence-based, not
# assumed.

print("Tokenizing every article (no truncation) to measure length distribution...")
_enc = finbert_tokenizer(
    news_df['text_combined'].tolist(),
    add_special_tokens=False,
    truncation=False,
    padding=False,
    return_attention_mask=False,
)
token_lengths = np.array([len(ids) for ids in _enc['input_ids']])
budget = HEAD_N + TAIL_N  # 510 content tokens (reserves 2 for [CLS], [SEP])

n = len(token_lengths)
pct_over = lambda cap: 100.0 * np.mean(token_lengths > cap)

print(f"\n=== Token-length distribution over {n:,} articles ===")
print(
    f"min / median / mean / max : "
    f"{token_lengths.min()} / {int(np.median(token_lengths))} / "
    f"{token_lengths.mean():.0f} / {token_lengths.max()}"
)
for q in (50, 75, 90, 95, 99, 99.9):
    print(f"  p{q:>5}: {int(np.percentile(token_lengths, q)):>6} tokens")

print("\n=== Share of articles exceeding each cap ===")
for cap in (512, 1024, 2048, 4096):
    n_over = int(np.sum(token_lengths > cap))
    print(f"  > {cap:>4} tokens: {n_over:>6,} articles ({pct_over(cap):5.2f}%)")

# Fraction of each article's content retained by head+tail.
kept_ratio = np.minimum(1.0, budget / np.maximum(token_lengths, 1))
long_mask = token_lengths > budget
pct_over_512 = pct_over(512)
pct_losing_half = 100.0 * np.mean(kept_ratio < 0.5)
pct_losing_three_quarters = 100.0 * np.mean(kept_ratio < 0.25)

if long_mask.any():
    print("\n=== Head+tail content retention (truncated articles) ===")
    print(f"  articles truncated                     : {long_mask.sum():,} / {n:,} "
          f"({100.0 * long_mask.mean():.2f}%)")
    print(f"  mean kept ratio among truncated         : {kept_ratio[long_mask].mean():.3f}")
    print(f"  median kept ratio among truncated       : {np.median(kept_ratio[long_mask]):.3f}")
    print(f"  articles losing > 50% of content        : {pct_losing_half:.2f}% of corpus")
    print(f"  articles losing > 75% of content        : {pct_losing_three_quarters:.2f}% of corpus")
else:
    print("\n=== No article exceeds the head+tail budget; no truncation happens. ===")

# --- Plots ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

ax1.hist(np.clip(token_lengths, 0, 5000), bins=80, color='steelblue', edgecolor='white')
for cap, color in [(512, 'orange'), (1024, 'crimson'), (2048, 'purple'), (4096, 'black')]:
    ax1.axvline(cap, color=color, linestyle='--', alpha=0.7,
                label=f'{cap} tokens ({pct_over(cap):.1f}% over)')
ax1.set_xlabel('Article length in WordPiece tokens (clipped at 5000 for display)')
ax1.set_ylabel('Number of articles')
ax1.set_title('Token-length distribution')
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3)

if long_mask.any():
    ax2.hist(kept_ratio[long_mask], bins=40, color='seagreen', edgecolor='white')
    ax2.axvline(0.5,  color='red',     linestyle='--', label='50% kept')
    ax2.axvline(0.25, color='darkred', linestyle='--', label='25% kept')
    ax2.set_xlabel('Fraction of article tokens kept by head+tail')
    ax2.set_ylabel('Number of truncated articles')
    ax2.set_title(f'Retention on the {long_mask.sum():,} truncated articles')
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.3)
else:
    ax2.set_visible(False)

plt.tight_layout()
save_fig('token_length_distribution.png')
plt.show()

# --- Rule-of-thumb recommendation ---
print("\n" + "=" * 60)
print("RECOMMENDATION")
print("=" * 60)
if pct_over_512 < 5.0:
    print(f"Only {pct_over_512:.2f}% of articles exceed 512 tokens.")
    print("-> Head+tail as implemented is more than sufficient.")
    print("   A chunk-pool fallback is not worth the added complexity.")
elif pct_over_512 < 20.0 and pct_losing_half < 5.0:
    print(f"{pct_over_512:.2f}% of articles exceed 512 tokens; only {pct_losing_half:.2f}% lose")
    print("more than half their content to head+tail.")
    print("-> Head+tail is fine. Minor signal loss on the upper tail, but a")
    print("   chunk-pool fallback likely isn't worth the complexity.")
elif pct_losing_half >= 10.0 or pct_losing_three_quarters >= 3.0:
    print(f"{pct_over_512:.2f}% of articles exceed 512 tokens; {pct_losing_half:.2f}% lose")
    print(f"more than half their content, and {pct_losing_three_quarters:.2f}% lose > 75%.")
    print("-> A chunk-pool fallback for very long articles is justified.")
    print("   Suggested next step: for articles above N*510 tokens (N~=3),")
    print("   split into 510-token windows, encode each, mean-pool the [CLS]")
    print("   vectors. Keep head+tail for everything shorter -- Sun et al.")
    print("   show it is strictly better on articles that fit comfortably.")
else:
    print(f"{pct_over_512:.2f}% of articles exceed 512 tokens; {pct_losing_half:.2f}% lose")
    print("more than half their content. Borderline case.")
    print("-> Head+tail is defensible. Revisit only if evaluation shows poor")
    print("   performance on the longest-article bucket.")


In [ ]:
# Score the selected text field with batched FinBERT inference.
sentiment_df = predict_sentiment_batch(news_df[TEXT_COL].tolist(), batch_size=16)
sentiment_df['finbert_label'] = sentiment_df[['neutral', 'positive', 'negative']].idxmax(axis=1)
sentiment_df['finbert_sentiment'] = sentiment_df['positive'] - sentiment_df['negative']
sentiment_df['finbert_text_source'] = TEXT_COL

# Make the cell safe to rerun when switching between title/body/combined inputs.
sentiment_cols = ['neutral', 'positive', 'negative', 'finbert_label', 'finbert_sentiment', 'finbert_text_source']
news_df = news_df.drop(columns=sentiment_cols, errors='ignore')

# Append the new columns to news_df
news_df = pd.concat([news_df.reset_index(drop=True), sentiment_df], axis=1)

print(news_df[['finbert_text_source', 'finbert_label', 'finbert_sentiment']].head())

In [ ]:
# save FinBERT sentiment
news_df.to_csv(SENTIMENT_PATH, index=False)


In [ ]:
news_df.tail()

## BERT Embeddings + MLP Return Model

Articles are encoded once with a frozen `bert-base-uncased` model (the [CLS] embedding of the head+tail-truncated title + body). A small MLP is then trained on those 768-dim embeddings to predict the mentioned stock's next-day return.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel

In [ ]:
# create the feature daily_return based on close price
price_df['daily_return'] = price_df.groupby('ticker')['close'].transform(lambda x: x.pct_change())
price_df = price_df.dropna()

# Convert dates to datetime
news_df['publication_datetime'] = pd.to_datetime(news_df['publication_datetime'])
price_df['Date'] = pd.to_datetime(price_df['Date'])

In [ ]:
from transformers import BertTokenizer, BertModel

# Initialize BERT tokenizer and model: https://huggingface.co/google-bert/bert-base-uncased
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased', use_safetensors=True)
bert_model.eval()  # Set to evaluation mode

In [ ]:
# Define the neural network
def get_activation(name):
    activation_map = {
        'relu': nn.ReLU,
        'gelu': nn.GELU,
        'tanh': nn.Tanh,
    }
    if name not in activation_map:
        raise ValueError(f"Unsupported activation: {name}")
    return activation_map[name]()

class ReturnPredictor(nn.Module):
    def __init__(self, input_size=768, hidden_dims=(256, 128), activation='relu', dropout=0.2):
        super(ReturnPredictor, self).__init__()
        layers = []
        previous_dim = input_size
        
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(previous_dim, hidden_dim))
            layers.append(get_activation(activation))
            layers.append(nn.Dropout(dropout))
            previous_dim = hidden_dim
        
        layers.append(nn.Linear(previous_dim, 1))
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

In [ ]:
# Precompute BERT embeddings for every article ONCE, then reuse them across
# every CV fold and the final retrain. This is what makes rolling/expanding
# window CV affordable -- the BERT forward pass is by far the expensive part.

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model.to(device)
bert_model.eval()

# (Date, ticker) -> daily_return lookup for O(1) access
return_map = price_df.set_index(['Date', 'ticker'])['daily_return'].to_dict()

# Per-ticker sorted trading-date arrays -> binary-search the "next trading day"
ticker_dates = {t: np.sort(g['Date'].values) for t, g in price_df.groupby('ticker')}

def next_day_return(pub_date, ticker):
    dates = ticker_dates.get(ticker)
    if dates is None or len(dates) == 0:
        return np.nan
    pos = np.searchsorted(dates, np.datetime64(pub_date), side='right')
    if pos >= len(dates):
        return np.nan
    next_date = pd.Timestamp(dates[pos])
    return return_map.get((next_date, ticker), np.nan)

# Attach supervised target to a working copy of news_df
news_df_full = news_df.copy()
news_df_full['publication_datetime'] = pd.to_datetime(news_df_full['publication_datetime'])
news_df_full['next_day_return'] = [
    next_day_return(d, t)
    for d, t in zip(news_df_full['publication_datetime'], news_df_full['tickers'])
]
news_df_full = news_df_full.dropna(subset=['next_day_return']).reset_index(drop=True)
print(f"Articles with valid next-day returns: {len(news_df_full)}")

# One-pass BERT embedding using head+tail truncation (Sun et al. 2019).
# Feed title + [SEP] + body so the article headline is always included.
print("Computing BERT embeddings for every article (head+tail truncation, reused across every CV fold)...")
news_df_full['title'] = news_df_full['title'].fillna('').astype(str)
news_df_full['body']  = news_df_full['body'].fillna('').astype(str)
combined_texts = (news_df_full['title'] + ' [SEP] ' + news_df_full['body']).str.strip().tolist()

all_embeddings = np.zeros((len(news_df_full), 768), dtype=np.float32)
_EMB_BATCH = 32
with torch.no_grad():
    for i in range(0, len(news_df_full), _EMB_BATCH):
        batch_texts = combined_texts[i:i+_EMB_BATCH]
        inputs = head_tail_tokenize(batch_texts, tokenizer,
                                    head_n=HEAD_N, tail_n=TAIL_N, max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = bert_model(**inputs)
        cls = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings[i:i+cls.shape[0]] = cls
        if (i // _EMB_BATCH) % 50 == 0:
            print(f"  processed {i}/{len(news_df_full)}")
print("Done computing embeddings.")

class EmbeddingDataset(Dataset):
    """Serves precomputed BERT [CLS] embeddings + next-day returns."""
    def __init__(self, embeddings, returns):
        self.embeddings = torch.FloatTensor(np.asarray(embeddings))
        self.returns = torch.FloatTensor(np.asarray(returns).reshape(-1, 1))
    def __len__(self):
        return len(self.embeddings)
    def __getitem__(self, idx):
        return self.embeddings[idx], self.returns[idx]

# Sort by date so expanding-window splits are chronological, and re-align the
# embedding matrix to the same sort order.
sort_order = np.argsort(news_df_full['publication_datetime'].values, kind='stable')
news_df_full_sorted = news_df_full.iloc[sort_order].reset_index(drop=True)
embeddings_sorted = all_embeddings[sort_order]

# Final holdout: pre-2020 is the CV pool (for rolling-window tuning + retraining),
# 2020 is locked away as the test set.
cv_mask = news_df_full_sorted['publication_datetime'] < '2020-01-01'
test_mask = ~cv_mask

cv_embeddings = embeddings_sorted[cv_mask.values]
cv_returns = news_df_full_sorted.loc[cv_mask, 'next_day_return'].values

test_embeddings = embeddings_sorted[test_mask.values]
test_returns = news_df_full_sorted.loc[test_mask, 'next_day_return'].values
test_news_df = news_df_full_sorted.loc[test_mask].reset_index(drop=True)

print(f"CV pool size (pre-2020): {len(cv_embeddings)}")
print(f"Test set size (2020+):   {len(test_embeddings)}")


In [ ]:
# Expanded hyperparameter grid. Each config varies multiple knobs simultaneously
# so the sweep covers width, depth, activation, dropout, learning rate, weight
# decay, and batch size. This replaces the earlier 3-config hand-pick.
experiment_configs = [
    {'name': 'baseline_relu',     'hidden_dims': (256, 128),     'activation': 'relu', 'dropout': 0.2, 'lr': 1e-3, 'weight_decay': 0.0,  'batch_size': 32,  'epochs': 15},
    {'name': 'wider_relu',        'hidden_dims': (512, 256),     'activation': 'relu', 'dropout': 0.3, 'lr': 5e-4, 'weight_decay': 1e-5, 'batch_size': 32,  'epochs': 15},
    {'name': 'deep_relu',         'hidden_dims': (256, 128, 64), 'activation': 'relu', 'dropout': 0.3, 'lr': 5e-4, 'weight_decay': 1e-5, 'batch_size': 32,  'epochs': 15},
    {'name': 'compact_tanh',      'hidden_dims': (256, 64),      'activation': 'tanh', 'dropout': 0.1, 'lr': 1e-3, 'weight_decay': 0.0,  'batch_size': 64,  'epochs': 15},
    {'name': 'gelu_medium',       'hidden_dims': (384, 128),     'activation': 'gelu', 'dropout': 0.2, 'lr': 1e-3, 'weight_decay': 1e-5, 'batch_size': 32,  'epochs': 15},
    {'name': 'high_dropout_relu', 'hidden_dims': (256, 128),     'activation': 'relu', 'dropout': 0.5, 'lr': 1e-3, 'weight_decay': 0.0,  'batch_size': 32,  'epochs': 15},
    {'name': 'low_lr_gelu',       'hidden_dims': (256, 128),     'activation': 'gelu', 'dropout': 0.2, 'lr': 1e-4, 'weight_decay': 1e-4, 'batch_size': 32,  'epochs': 20},
    {'name': 'large_batch_relu',  'hidden_dims': (512, 256),     'activation': 'relu', 'dropout': 0.3, 'lr': 2e-3, 'weight_decay': 1e-5, 'batch_size': 128, 'epochs': 15},
    {'name': 'wd_heavy_tanh',     'hidden_dims': (256, 128),     'activation': 'tanh', 'dropout': 0.2, 'lr': 5e-4, 'weight_decay': 1e-3, 'batch_size': 32,  'epochs': 15},
]
print(f"Running a hyperparameter sweep over {len(experiment_configs)} configurations.")


In [ ]:
# Expanding-window time-series CV (TimeSeriesSplit) for hyperparameter selection.
# With n_splits=4 each fold's training window grows and its validation window
# marches forward in time -- no future data leaks into the past, which is the
# whole point for a market-prediction task.
from sklearn.model_selection import TimeSeriesSplit

N_SPLITS = 4
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

def run_one_fold(config, train_emb, train_ret, val_emb, val_ret):
    train_loader = DataLoader(EmbeddingDataset(train_emb, train_ret),
                              batch_size=config['batch_size'], shuffle=True)
    val_loader = DataLoader(EmbeddingDataset(val_emb, val_ret),
                            batch_size=config['batch_size'])
    torch.manual_seed(42)
    model_ = ReturnPredictor(
        input_size=train_emb.shape[1],
        hidden_dims=config['hidden_dims'],
        activation=config['activation'],
        dropout=config['dropout'],
    ).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model_.parameters(),
                                 lr=config['lr'],
                                 weight_decay=config.get('weight_decay', 0.0))

    tr_curve, v_curve = [], []
    best_val, best_state = float('inf'), None
    for _ in range(config['epochs']):
        model_.train()
        running = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model_(xb), yb)
            loss.backward()
            optimizer.step()
            running += loss.item()
        tr_curve.append(running / max(1, len(train_loader)))

        model_.eval()
        vrun = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                vrun += criterion(model_(xb), yb).item()
        v_curve.append(vrun / max(1, len(val_loader)))

        if v_curve[-1] < best_val:
            best_val = v_curve[-1]
            best_state = {k: v.detach().cpu().clone() for k, v in model_.state_dict().items()}

    return tr_curve, v_curve, best_val, best_state


print("=" * 60)
print("EXPANDING-WINDOW CV HYPERPARAMETER SEARCH")
print("=" * 60)
cv_results = {}
for config in experiment_configs:
    print(f"\n>> Config: {config['name']}")
    fold_tr, fold_v, fold_best = [], [], []
    for fold, (tr_pos, val_pos) in enumerate(tscv.split(cv_embeddings), start=1):
        tr_curve, v_curve, best_val, _ = run_one_fold(
            config,
            cv_embeddings[tr_pos], cv_returns[tr_pos],
            cv_embeddings[val_pos], cv_returns[val_pos],
        )
        fold_tr.append(tr_curve)
        fold_v.append(v_curve)
        fold_best.append(best_val)
        print(f"   fold {fold}: train_n={len(tr_pos)}, val_n={len(val_pos)}, best_val_mse={best_val:.6f}")
    cv_results[config['name']] = {
        'config': config,
        'fold_train_curves': fold_tr,
        'fold_val_curves': fold_v,
        'fold_best_vals': fold_best,
        'mean_val_mse': float(np.mean(fold_best)),
        'std_val_mse': float(np.std(fold_best)),
    }

cv_summary = pd.DataFrame([
    {
        'name': name,
        'mean_val_mse': res['mean_val_mse'],
        'std_val_mse': res['std_val_mse'],
        **{k: v for k, v in res['config'].items() if k != 'name'},
    }
    for name, res in cv_results.items()
]).sort_values('mean_val_mse').reset_index(drop=True)

print("\nCV summary (ranked by mean validation MSE across folds):")
print(cv_summary.to_string(index=False))

best_config_name = cv_summary.loc[0, 'name']
best_config = cv_results[best_config_name]['config']
print(f"\nBest configuration by CV: {best_config_name}")
print(best_config)

# Retrain on the full pre-2020 pool with the winning config, holding out the
# last 10% as a validation slice for early stopping + the final learning curve.
split_point = int(0.9 * len(cv_embeddings))
final_train_emb, final_train_ret = cv_embeddings[:split_point], cv_returns[:split_point]
final_val_emb,   final_val_ret   = cv_embeddings[split_point:], cv_returns[split_point:]
train_losses, val_losses, best_val_final, best_state = run_one_fold(
    best_config, final_train_emb, final_train_ret, final_val_emb, final_val_ret
)

model = ReturnPredictor(
    input_size=final_train_emb.shape[1],
    hidden_dims=best_config['hidden_dims'],
    activation=best_config['activation'],
    dropout=best_config['dropout'],
).to(device)
model.load_state_dict(best_state)
model.eval()
print(f"\nFinal retrain with {best_config_name}: best val MSE = {best_val_final:.6f}")

# Test loader for the locked-away 2020 holdout
test_loader = DataLoader(
    EmbeddingDataset(test_embeddings, test_returns),
    batch_size=best_config['batch_size'],
)


In [ ]:
# Learning curves for every configuration across every CV fold. Each subplot
# overlays the folds' train/val curves so you can see fold-to-fold variance
# and spot overfitting. The bottom plot shows the final retrain curve.
n_cfg = len(experiment_configs)
cols = 3
rows = int(np.ceil(n_cfg / cols))
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 3.5 * rows), sharey=True)
axes = np.array(axes).reshape(-1)

for ax, config in zip(axes, experiment_configs):
    res = cv_results[config['name']]
    for fold_idx, (tc, vc) in enumerate(zip(res['fold_train_curves'], res['fold_val_curves'])):
        ax.plot(tc, color='steelblue', alpha=0.6, label='Train' if fold_idx == 0 else None)
        ax.plot(vc, color='orange',    alpha=0.6, label='Val'   if fold_idx == 0 else None)
    ax.set_title(
        f"{config['name']}\nmean val MSE = {res['mean_val_mse']:.4f} (±{res['std_val_mse']:.4f})",
        fontsize=10,
    )
    ax.set_xlabel('Epoch')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
for ax in axes[n_cfg:]:
    ax.set_visible(False)
axes[0].set_ylabel('MSE')
plt.suptitle('Learning curves across expanding-window CV folds', y=1.02)
plt.tight_layout()
save_fig('cv_learning_curves.png')
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(train_losses, label='Train', color='steelblue')
plt.plot(val_losses,   label='Val',   color='orange')
plt.title(f"Final retrain learning curve ({best_config_name})")
plt.xlabel('Epoch'); plt.ylabel('MSE'); plt.legend(); plt.grid(alpha=0.3)
save_fig('final_learning_curve.png')
plt.show()


In [ ]:
def calculate_r2(y_true, y_pred, in_sample=True, benchmark=None):
    if in_sample:
        return 1 - (np.sum((y_true - y_pred) ** 2) / 
                    np.sum((y_true - np.mean(y_true)) ** 2))
    else:
        if benchmark is None:
            raise ValueError("Benchmark must be provided for out-of-sample R-squared calculation.")
        return 1 - (np.sum((y_true - y_pred) ** 2) / 
                    np.sum((y_true - benchmark) ** 2))

In [ ]:
# Evaluate the retrained best model on the 2020 holdout test set.
model.eval()
test_predictions, test_actuals = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        test_predictions.extend(model(xb).cpu().numpy())
        test_actuals.extend(yb.numpy())
test_predictions = np.array(test_predictions).reshape(-1)
test_actuals = np.array(test_actuals).reshape(-1)

r2 = calculate_r2(test_actuals, test_predictions, in_sample=False, benchmark=0)
print(f"Best configuration: {best_config_name}")
print(f"Test R2 Score: {r2:.6f}")


In [ ]:
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(test_actuals, test_predictions)
print (f'Test MSE: {mse:.6f}')

In [ ]:
# Build the aligned 2020 test frame. FinBERT gives a 3-way softmax directly;
# the MLP gives a continuous return forecast which we map to a 3-class signal
# (down / neutral / up) using a data-driven dead-zone.

news_df_test_base = test_news_df.copy()
news_df_test_base['mlp_predicted_return'] = test_predictions
news_df_test_base['actual_return'] = test_actuals

# Join back the 3-way FinBERT probabilities for these articles.
nw = pd.read_csv(SENTIMENT_PATH)
nw['publication_datetime'] = pd.to_datetime(nw['publication_datetime'])
nw_unique = nw.drop_duplicates(subset=['publication_datetime', 'title', 'tickers'], keep='first')
news_df_test_base = news_df_test_base.merge(
    nw_unique[['publication_datetime', 'title', 'tickers', 'neutral', 'positive', 'negative']],
    on=['publication_datetime', 'title', 'tickers'],
    how='left',
)
# If any merge misses happen (duplicate titles etc.), fall back to a uniform prior.
news_df_test_base[['neutral', 'positive', 'negative']] = (
    news_df_test_base[['neutral', 'positive', 'negative']].fillna(1.0 / 3.0)
)
news_df_test_base['finbert_sentiment'] = news_df_test_base['positive'] - news_df_test_base['negative']

# --- Neutral-aware 3-class labels ---------------------------------------------
# Dead-zone tau = 0.25 * std(cv_returns). Anything inside ±tau is "neutral".
tau = 0.25 * float(np.std(cv_returns))
print(f"Neutral band: |return| < tau with tau = {tau:.6f}")

def three_class_from_return(vals, t):
    return np.where(vals > t, 'up', np.where(vals < -t, 'down', 'neutral'))

news_df_test_base['actual_class']        = three_class_from_return(news_df_test_base['actual_return'].values, tau)
news_df_test_base['mlp_predicted_class'] = three_class_from_return(news_df_test_base['mlp_predicted_return'].values, tau)

# FinBERT has a native 3-class head: pick argmax over (neutral, positive, negative)
def finbert_argmax_class(row):
    probs = {'neutral': row['neutral'], 'up': row['positive'], 'down': row['negative']}
    return max(probs, key=probs.get)
news_df_test_base['finbert_class'] = news_df_test_base.apply(finbert_argmax_class, axis=1)

# Backward-compat columns used by downstream monthly/market-alignment cells.
news_df_test_base['mlp_directional_signal'] = np.where(news_df_test_base['mlp_predicted_return'] > 0, 1, -1)
news_df_test_base['actual_direction']       = np.where(news_df_test_base['actual_return'] > 0, 1, -1)


In [ ]:
news_df_test_base.head()

## Sentiment and Market Trend

### Pretrained FinBERT

In [ ]:
# read the news data with FinBERT sentiment
news_w_sentiment = pd.read_csv(SENTIMENT_PATH)

news_w_sentiment.head()

In [ ]:
# create monthly FinBERT sentiment on the aligned 2020 test set
news_w_sentiment['sentiment'] = news_w_sentiment['positive'] - news_w_sentiment['negative']
news_df_test_finbert = news_df_test_base.copy()

# monthly sentiment for FinBERT
monthly_finbert = news_df_test_finbert.groupby(news_df_test_finbert['publication_datetime'].dt.to_period('M'))['finbert_sentiment'].mean().reset_index()
monthly_finbert.columns = ['month', 'sentiment']

In [ ]:
def calculate_sp_return(month):
    current_period = price_df.loc[
        (price_df['Date'] >= month.start_time) & 
        (price_df['Date'] < (month + 1).start_time) & 
        (price_df['ticker'] == 'SPX')
    ]
    
    next_period = price_df.loc[
        (price_df['Date'] >= (month + 1).start_time) & 
        (price_df['Date'] < (month + 2).start_time) & 
        (price_df['ticker'] == 'SPX')
    ]
    
    if not current_period.empty and not next_period.empty:
        return (next_period['close'].iloc[-1] / current_period['close'].iloc[-1]) - 1
    else:
        return None

In [ ]:
# monthly return for S&P500; the return is calculated based on the close price at the last trading date of the current month and that of the next month
monthly_finbert['SP_return'] = monthly_finbert['month'].apply(calculate_sp_return)

In [ ]:
# display the result
monthly_finbert

In [ ]:
from sklearn.linear_model import LinearRegression

X_finbert = monthly_finbert['sentiment'].values.reshape(-1, 1)
y_finbert = monthly_finbert['SP_return'].values

reg_finbert = LinearRegression().fit(X_finbert, y_finbert)
r2_finbert = reg_finbert.score(X_finbert, y_finbert)

print(f"FinBERT - R²: {r2_finbert:.4f}")
print(f"Coefficient: {reg_finbert.coef_[0]:.4f}")

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(monthly_finbert['sentiment'], monthly_finbert['SP_return'], color='blue', alpha=0.5)

x_range = np.array([monthly_finbert['sentiment'].min(), monthly_finbert['sentiment'].max()])
plt.plot(x_range, reg_finbert.predict(x_range.reshape(-1, 1)), 'r-', label=f'R² = {r2_finbert:.3f}')

plt.title('FinBERT: Sentiment vs S&P 500 Return')
plt.xlabel('Sentiment')
plt.ylabel('S&P 500 Return')
plt.legend()
plt.grid(True)
plt.axhline(0, color='red', linestyle='--')
save_fig('finbert_vs_sp500_scatter.png')
plt.show()

In [ ]:
# Time-series plot of the MLP directional signal against market return.
fig, ax1 = plt.subplots(figsize=(12, 6))

# Convert 'month' to a string format for plotting
ax1.set_xlabel('Month')
ax1.set_ylabel('Sentiment', color='blue')
ax1.plot(monthly_finbert['month'].astype(str), monthly_finbert['sentiment'], label='Sentiment', color='blue', marker='o')
ax1.tick_params(axis='y', labelcolor='blue')

# Create a second y-axis for the S&P 500 return
ax2 = ax1.twinx()  
ax2.set_ylabel('S&P 500 Return', color='orange')  
ax2.plot(monthly_finbert['month'].astype(str), monthly_finbert['SP_return'], label='S&P 500 Return', color='orange', marker='x')
ax2.tick_params(axis='y', labelcolor='orange')

plt.title('Time Series of Sentiment and S&P 500 Return')
fig.tight_layout()  
plt.grid(True)
plt.xticks(rotation=45)
save_fig('finbert_vs_sp500_timeseries.png')
plt.show()

### BERT + MLP Model


In [ ]:
# Create monthly MLP-derived market signals on the same aligned 2020 test set.
# The raw model output is a predicted return, so use a directional signal for the
# market-alignment plots and keep the mean predicted return for reference.
news_df_test_mlp = news_df_test_base.copy()
monthly_mlp = news_df_test_mlp.groupby(news_df_test_mlp['publication_datetime'].dt.to_period('M')).agg(
    mean_predicted_return=('mlp_predicted_return', 'mean'),
    bullish_share=('mlp_predicted_return', lambda x: (x > 0).mean()),
    directional_signal=('mlp_directional_signal', 'mean')
).reset_index()
monthly_mlp.columns = ['month', 'mean_predicted_return', 'bullish_share', 'directional_signal']

In [ ]:
# monthly return for S&P500; the return is calculated based on the close price at the last trading date of the current month and that of the next month
monthly_mlp['SP_return'] = monthly_mlp['month'].apply(calculate_sp_return)

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

# Regression for the MLP-derived directional signal.
X_mlp = monthly_mlp['directional_signal'].values.reshape(-1, 1)
y_mlp = monthly_mlp['SP_return'].values
reg_mlp = LinearRegression().fit(X_mlp, y_mlp)
r2_mlp = reg_mlp.score(X_mlp, y_mlp)

print(f"BERT+MLP directional signal - R²: {r2_mlp:.4f}")
print(f"Coefficient: {reg_mlp.coef_[0]:.4f}")
print(monthly_mlp[['month', 'mean_predicted_return', 'bullish_share', 'directional_signal', 'SP_return']])

In [ ]:
# Scatter plot with regression line for the BERT+MLP directional signal.
plt.figure(figsize=(10, 6))
plt.scatter(monthly_mlp['directional_signal'], monthly_mlp['SP_return'], 
            color='blue', alpha=0.6, label='Monthly data points')

x_range = np.array([monthly_mlp['directional_signal'].min(), monthly_mlp['directional_signal'].max()])
y_pred = reg_mlp.predict(x_range.reshape(-1, 1))
plt.plot(x_range, y_pred, 'r-', linewidth=2, label=f'Regression line (R² = {r2_mlp:.3f})')

plt.title('BERT+MLP: Directional Signal vs S&P 500 Return')
plt.xlabel('Monthly mean directional signal (-1 bearish, +1 bullish)')
plt.ylabel('S&P 500 Return')
plt.grid(True)
plt.axhline(0, color='red', linestyle='--', alpha=0.5)
plt.axvline(0, color='gray', linestyle='--', alpha=0.5)
plt.legend()
save_fig('mlp_vs_sp500_scatter.png')
plt.show()

In [ ]:
# Time-series plot of the MLP directional signal against market return.
fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.set_xlabel('Month')
ax1.set_ylabel('MLP directional signal', color='blue')
ax1.plot(monthly_mlp['month'].astype(str), monthly_mlp['directional_signal'], label='MLP directional signal', color='blue', marker='o')
ax1.tick_params(axis='y', labelcolor='blue')
ax1.axhline(0, color='blue', linestyle='--', alpha=0.3)

ax2 = ax1.twinx()
ax2.set_ylabel('S&P 500 Return', color='orange')
ax2.plot(monthly_mlp['month'].astype(str), monthly_mlp['SP_return'], label='S&P 500 Return', color='orange', marker='x')
ax2.tick_params(axis='y', labelcolor='orange')
ax2.axhline(0, color='orange', linestyle='--', alpha=0.3)

plt.title('BERT+MLP: Time Series of Directional Signal and S&P 500 Return')
fig.tight_layout()
plt.grid(True)
plt.xticks(rotation=45)
save_fig('mlp_vs_sp500_timeseries.png')
plt.show()

In [ ]:
# Verify that FinBERT sentiment, the MLP return forecast, and actual returns are aligned.
print(news_df_test_base[['publication_datetime', 'tickers', 'finbert_sentiment', 'mlp_predicted_return', 'mlp_directional_signal', 'actual_return']].head())

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import seaborn as sns

# Three-class evaluation (down / neutral / up). The MLP's continuous return
# forecast is mapped to a class using the same ±tau dead-zone as the ground
# truth. FinBERT uses its native 3-way softmax.
labels = ['down', 'neutral', 'up']
y_true  = news_df_test_base['actual_class'].values
mlp_pred = news_df_test_base['mlp_predicted_class'].values
fb_pred  = news_df_test_base['finbert_class'].values

print("=" * 55)
print("THREE-CLASS CLASSIFICATION RESULTS (down / neutral / up)")
print("=" * 55)

for model_name, y_pred in [('FinBERT', fb_pred), ('BERT+MLP', mlp_pred)]:
    acc      = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, labels=labels, average='macro', zero_division=0)
    print(f"\n{model_name}")
    print(f"  Accuracy: {acc:.4f}   Macro F1: {f1_macro:.4f}")
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels)
    plt.title(f'{model_name} Confusion Matrix (3-class)')
    plt.xlabel('Predicted'); plt.ylabel('Actual')
    save_fig(f"confusion_matrix_{model_name.lower().replace(' ', '_').replace('+', '')}.png")
    plt.show()

# Also keep the original binary summary for reference / backward comparison
y_true_binary  = (news_df_test_base['actual_return'].values > 0).astype(int)
finbert_binary = (news_df_test_base['finbert_sentiment'].values > 0).astype(int)
mlp_binary     = (news_df_test_base['mlp_predicted_return'].values > 0).astype(int)
print("\n--- Binary (up/down only, for reference) ---")
print(f"FinBERT         acc={accuracy_score(y_true_binary, finbert_binary):.4f}  f1={f1_score(y_true_binary, finbert_binary):.4f}")
print(f"BERT+MLP  acc={accuracy_score(y_true_binary, mlp_binary):.4f}  f1={f1_score(y_true_binary, mlp_binary):.4f}")


In [ ]:
print("\n" + "="*50)
print("MARKET TREND ALIGNMENT COMPARISON")
print("="*50)
print(f"{'Model':<20} {'R²':<12}")
print(f"{'-'*35}")
print(f"{'Pretrained FinBERT':<20} {r2_finbert:<12.4f}")
print(f"{'MLP directional signal':<20} {r2_mlp:<12.4f}")